<a href="https://colab.research.google.com/github/ragiokay/ML-2026-HW7/blob/main/ML2026_Spring_HW7_Model_Merging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ML2026 Spring HW7 — Model Merging**

在這份作業中，將實作 **Model Merging** 。

我們會使用兩個 7B 模型：
- `augmxnt/shisa-gamma-7b-v1`（擅長日文）
- `WizardLMTeam/WizardMath-7B-V1.1`（擅長數學推理）

**❗重要規則：**

- **請勿使用上述兩個模型以外的任何其他模型。**
- 你可以自由使用任何套件（不限於 mergekit）或自己撰寫的演算法來進行 merging。
- **Merging 的嚴格定義：合併後的模型總參數量必須與單一 base model 相同。** 換言之，MoE（Mixture of Experts）、Ensemble、Stacking 等會使 inference 時可使用的參數量增加的做法，皆視為違規。

你的目標是：
1. 先觀察兩個 base model 各自的能力
2. 嘗試各種 merge 方法，把兩個模型合併
3. 在日文數學題上測試合併後的模型表現，並將結果上傳到 judgeboi

# **Section 0: Preparing**

## Mount your notebook to your Drive

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Activate GPU
Model Merging 與 Inference 都需要 GPU，請務必啟用。

### **MUST READ**:

Colab does **NOT** guarantee the GPU access for free user ([ref](https://research.google.com/colaboratory/faq.html#idle-timeouts)). It is possible you get a message saying "Cannot connect to GPU backend" which means there are no enough GPU resources for you now. When this happens, you may need to **wait for one (or more) day or login different Google account to do the homework**.

### Enable GPU

1. Click on "Runtime" (or "執行階段") in the header.
2. Click on "Change runtime type" (or "變更執行階段類型") in the drop menu.
3. Select "T4 GPU" and save. (You can select "A100 GPU" or "V100 GPU" if you have Colab Pro)

## Check the status of your GPU

In [ ]:
!nvidia-smi

Wed May 20 00:12:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install Packages & Download Data

安裝 `mergekit`（模型合併工具）及下載日文數學題目檔案。

In [ ]:
# Install mergekit
!pip install -q mergekit

# Install / upgrade gdown for Google Drive download
!pip install -U -q gdown

# Download the Japanese math question file
!gdown 1_9OAxDwQgz1jUnlhHlkk3lLCTpY1Ib8Q -O /content/jp_math_question.json
# Verify download
import json
with open("/content/jp_math_question.json", "r") as f:
    questions = json.load(f)
print(f"成功載入 {len(questions)} 題日文數學題目！")
print(f"範例題目: {questions[0]['question'][:80]}...")
print(f"範例答案: {questions[0]['answer_number']}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 46.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mcp 1.27.0 requires pydantic<3.0.0,>=2.11.0, but you have pydantic 2.10.6 which is incompatible.
googl

# **Section 1: Explore Base Models — 觀察兩個 Base Model 的能力**
## 若時間不足可以先跳到Section 2

在合併之前，我們先分別載入兩個 base model，觀察它們各自的強項與弱項。

- **shisa-gamma-7b-v1**：以日文能力見長的模型
- **WizardMath-7B-V1.1**：以數學推理見長的模型

我們會各丟一個 **日文問題** 和一個 **數學問題** 給兩個模型，直覺感受它們的差異。

## 1.1 定義推論用的 helper function

建立通用的推論與評估函數，後續兩個模型都會用到。

In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    """通用推論函數：給定模型與 prompt，回傳生成結果"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1
        )
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_output[len(prompt):].strip()

In [ ]:
import re

############################################################
# Prompt 模板
USER_PROMPT_TEMPLATE = (
    "次の数学の問題を解いて、最終的な数値の答えを提示してください。\n\n"
    "問題: {question}\n\n"
    "ステップバイステップで考えて、最後に『答え: [数値]』の形式で答えを提示してください。"
)

# ⛔ 嚴禁修改 USER_PROMPT_TEMPLATE！
#
# 本作業聚焦於 Model Merging 技術本身，
# 而非 Prompt Engineering，因此為確保所有同學的
# 評測條件一致，無論任何情況皆請勿修改以下 prompt。
# 違反此規則將視為違規。
############################################################


def extract_number(text):
    """從模型輸出中提取數字答案"""
    match = re.search(r'答え\s*[:：]\s*(\d+)', text)
    if match:
        return match.group(1)
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else None

def evaluate_model(model, tokenizer, questions, model_name):
    """在日文數學題集上評估模型"""
    correct = 0
    results = []
    print(f"\n{'='*60}")
    print(f"📊 評估模型: {model_name}")
    print(f"{'='*60}\n")

    for i, item in enumerate(questions):
        prompt = USER_PROMPT_TEMPLATE.format(question=item['question'])
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                repetition_penalty=1.1
            )
        full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer_part = full_output[len(prompt):].strip()
        predicted = extract_number(answer_part)
        expected = item["answer_number"]
        is_correct = predicted == expected
        if is_correct:
            correct += 1
        results.append({
            "id": i + 1, "correct": is_correct,
            "predicted": predicted, "expected": expected,
            "raw": answer_part,
        })
        mark = "✅" if is_correct else "❌"
        print(f"{mark} Q{i+1:02d} | 預測: {str(predicted):>6s} | 正解: {expected:>6s}")
        print(f"     | 題目: {item['question'][:60]}")
        print(f"     | raw: {answer_part[:120]}")
        print()

    print(f"{'='*50}")
    print(f"🎯 {model_name} 正確率: {correct}/{len(questions)} ({correct/len(questions)*100:.1f}%)")
    print(f"{'='*50}")
    return correct, results

## 1.2 測試 Base Model ①：shisa-gamma-7b-v1（日文模型）

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("載入 shisa-gamma-7b-v1 ...")
shisa_name = "augmxnt/shisa-gamma-7b-v1"
shisa_tokenizer = AutoTokenizer.from_pretrained(shisa_name)
shisa_model = AutoModelForCausalLM.from_pretrained(
    shisa_name, torch_dtype=torch.float16, device_map="auto"
)
print("shisa-gamma-7b-v1 載入完成！")

載入 shisa-gamma-7b-v1 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

shisa-gamma-7b-v1 載入完成！


### 日文能力測試（shisa）

In [ ]:
jp_prompt = "日本の四季の特徴について、それぞれ簡潔に説明してください。"

print("=" * 60)
print("📝 日文 Prompt:", jp_prompt)
print("=" * 60)
print("\n🔵 【shisa-gamma-7b（日文模型）】的回答：")
print("-" * 40)
shisa_jp_ans = generate_response(shisa_model, shisa_tokenizer, jp_prompt)
print(shisa_jp_ans[:500])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 日文 Prompt: 日本の四季の特徴について、それぞれ簡潔に説明してください。

🔵 【shisa-gamma-7b（日文模型）】的回答：
----------------------------------------
また、それらを感じることができる場所やイベントなども教えてください。


### 數學能力測試（shisa）

In [ ]:
math_prompt = (
    "Solve the following math problem step by step.\n\n"
    "Problem: A store sells apples for $2 each and oranges for $3 each. "
    "If a customer buys 5 apples and 4 oranges, and pays with a $50 bill, "
    "how much change should the customer receive?\n\n"
    "Solution:"
)

print("=" * 60)
print("📝 Math Prompt:")
print(math_prompt)
print("=" * 60)
print("\n🔵 【shisa-gamma-7b（日文模型）】的回答：")
print("-" * 40)
shisa_math_ans = generate_response(shisa_model, shisa_tokenizer, math_prompt)
print(shisa_math_ans[:500])

print("\n" + "=" * 60)
print("💡 正確答案：$50 - (5×$2 + 4×$3) = $50 - $22 = $28")
print("=" * 60)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Math Prompt:
Solve the following math problem step by step.

Problem: A store sells apples for $2 each and oranges for $3 each. If a customer buys 5 apples and 4 oranges, and pays with a $50 bill, how much change should the customer receive?

Solution:

🔵 【shisa-gamma-7b（日文模型）】的回答：
----------------------------------------
1. First, let's calculate the total cost of the fruits purchased by the customer.
   Total cost = Price of apples * Number of apples + Price of oranges * Number of oranges
   Total cost = $2/apple * 5 apples + $3/orange * 4 oranges
   Total cost = $10 + $12
   Total cost = $22

2. Now, we need to find out how much money the customer has spent. Since they paid with a $50 bill, they have spent $50.

3. Finally, we can calculate the amount of change by subtracting the total cost from the amount spe

💡 正確答案：$50 - (5×$2 + 4×$3) = $50 - $22 = $28


### 在日文數學題集上評估（shisa）

In [ ]:
with open("/content/jp_math_question.json", "r") as f:
    questions = json.load(f)

shisa_correct, shisa_results = evaluate_model(
    shisa_model, shisa_tokenizer, questions, "shisa-gamma-7b（日文模型）"
)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



📊 評估模型: shisa-gamma-7b（日文模型）



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


❌ Q01 | 預測:     37 | 正解:     27
     | 題目: 太郎は平成5年に生まれました。次郎は令和2年に生まれました。太郎は次郎より何歳年上ですか？
     | raw: 1. まず、両方の人物が生まれた年を調べましょう。
    - 太郎は平成5年に生まれました。
    - 次郎は令和2年に生まれました。
2. これらの年を比較してみましょう。
    - 平成5年 = 西暦1993年
    - 令和2



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

❌ Q17 | 預測:     22 | 正解:   2036
     | 題目: 2024年は辰年（龍）です。干支は12年で一周します。次に辰年が来るのは西暦何年ですか？
     | raw: 1. 干支は12年で一周します。つまり、12年ごとに同じ干支が繰り返されます。
2. 2024年は辰年です。
3. この12年間で、もう一度辰年が来るまでにあと何年必要かを知る必要があります。
4. そのためには、2024年以降の各年がどの



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


✅ Q18 | 預測:   1400 | 正解:   1400
     | 題目: 回転寿司で、赤い皿は100円、金の皿は300円です。ある客が赤い皿を5枚、金の皿を3枚食べました。消費税を計算する前の合
     | raw: 1. まず、赤い皿の代金を計算します。
   赤い皿 = 100 * 5 = 500円

2. 次に、金の皿の代金を計算します。
   金の皿 = 300 * 3 = 900円

3. これらの代金を足して、合計金額を求めます。
   合計



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


❌ Q19 | 預測:      7 | 正解:     11
     | 題目: 日本の伝統的な行事である「節分」は2月3日です。バレンタインデーは2月14日です。節分の日からバレンタインデーまでは何日
     | raw: 1. 節分の日付を調べます。今年は2021年です。
2. 2021年の2月3日を見つけます。
3. それから、バレンタインデーの日付を調べます。今年は2021年です。
4. 2021年の2月14日を見つけます。
5. 節分の日からバレンタイ

❌ Q20 | 預測:      7 | 正解:      8
     | 題目: 日本の主要4島は本州、北海道、九州、四国です。本州が5つの地方に分けられ、残りの3島がそれぞれ1つの地方であるとした場合
     | raw: 1. まず、各島に割り当てられた地方の数を調べます。
2. 本州は5つの地方に分けられていることがわかっているので、本州には5つの地方があります。
3. 北海道、九州、四国はそれぞれ1つの地方に属しているので、それぞれ1つの地方があります。

🎯 shisa-gamma-7b（日文模型） 正確率: 8/20 (40.0%)


### 釋放 shisa 模型記憶體

In [ ]:
import gc

del shisa_model, shisa_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("已釋放 shisa-gamma-7b 記憶體！")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv

已釋放 shisa-gamma-7b 記憶體！
memory.used [MiB], memory.free [MiB]
542 MiB, 39900 MiB


## 1.3 測試 Base Model ②：WizardMath-7B-V1.1（數學模型）

In [ ]:
print("載入 WizardMath-7B-V1.1 ...")
wizard_name = "WizardLMTeam/WizardMath-7B-V1.1"
wizard_tokenizer = AutoTokenizer.from_pretrained(wizard_name)
wizard_model = AutoModelForCausalLM.from_pretrained(
    wizard_name, torch_dtype=torch.float16, device_map="auto"
)
print("WizardMath-7B-V1.1 載入完成！")

載入 WizardMath-7B-V1.1 ...


config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/948 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

WizardMath-7B-V1.1 載入完成！


### 日文能力測試（WizardMath）

In [ ]:
print("=" * 60)
print("📝 日文 Prompt:", jp_prompt)
print("=" * 60)
print("\n🟠 【WizardMath-7B（數學模型）】的回答：")
print("-" * 40)
wizard_jp_ans = generate_response(wizard_model, wizard_tokenizer, jp_prompt)
print(wizard_jp_ans[:500])

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 日文 Prompt: 日本の四季の特徴について、それぞれ簡潔に説明してください。

🟠 【WizardMath-7B（數學模型）】的回答：
----------------------------------------
The four seasons in Japan have distinct characteristics that set them apart from each other. Here is a brief explanation of each season:

1. Spring (春, Haru): Spring in Japan typically starts in March and lasts until May. It is a time of renewal and rebirth, as the cherry blossoms (sakura) bloom across the country. The weather during spring is generally mild, with temperatures ranging from 10 to 20 degrees Celsius.

2. Summer (夏, Natsu): Summer in Japan begins in June and ends in August. It is c


### 數學能力測試（WizardMath）

In [ ]:
print("=" * 60)
print("📝 Math Prompt:")
print(math_prompt)
print("=" * 60)
print("\n🟠 【WizardMath-7B（數學模型）】的回答：")
print("-" * 40)
wizard_math_ans = generate_response(wizard_model, wizard_tokenizer, math_prompt)
print(wizard_math_ans[:500])

print("\n" + "=" * 60)
print("💡 正確答案：$50 - (5×$2 + 4×$3) = $50 - $22 = $28")
print("=" * 60)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📝 Math Prompt:
Solve the following math problem step by step.

Problem: A store sells apples for $2 each and oranges for $3 each. If a customer buys 5 apples and 4 oranges, and pays with a $50 bill, how much change should the customer receive?

Solution:

🟠 【WizardMath-7B（數學模型）】的回答：
----------------------------------------
First, we need to calculate the total cost of the apples and oranges that the customer bought. 

The cost of the apples is calculated by multiplying the price per apple ($2) by the number of apples (5). So, the cost of the apples is $2 * 5 = $10.

Next, we calculate the cost of the oranges in the same way. The price per orange is $3 and the customer bought 4 oranges. So, the cost of the oranges is $3 * 4 = $12.

Now, we add the cost of the apples and the oranges together to find the total cost o

💡 正確答案：$50 - (5×$2 + 4×$3) = $50 - $22 = $28


### 在日文數學題集上評估（WizardMath）

In [ ]:
wizard_correct, wizard_results = evaluate_model(
    wizard_model, wizard_tokenizer, questions, "WizardMath-7B（數學模型）"
)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



📊 評估模型: WizardMath-7B（數學模型）



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


❌ Q01 | 預測:     25 | 正解:     27
     | 題目: 太郎は平成5年に生まれました。次郎は令和2年に生まれました。太郎は次郎より何歳年上ですか？
     | raw: 1. 今は平成30年です。
2. 太郎は平成5年に生まれました。
3. 今の年は平成30-平成5=25年経過しています。
4. 次郎は令和2年に生まれました。
5. 今の年は令和2+29=31年経過しています。
6. 太郎は25年経過してい



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

❌ Q17 | 預測:   2024 | 正解:   2036
     | 題目: 2024年は辰年（龍）です。干支は12年で一周します。次に辰年が来るのは西暦何年ですか？
     | raw: 1. 2024年は辰年（龍）です。
2. 干支は12年で一周します。
3. 辰年が来るのは西暦何年ですか？

To solve this problem, we need to determine when the next year of



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


✅ Q18 | 預測:   1400 | 正解:   1400
     | 題目: 回転寿司で、赤い皿は100円、金の皿は300円です。ある客が赤い皿を5枚、金の皿を3枚食べました。消費税を計算する前の合
     | raw: 1. 初めに、赤い皿の価格と金の皿の価格を定義します。
   - 赤い皿の価格: 100円
   - 金の皿の価格: 300円

2. 次に、客が食べた皿の枚数を定義します。
   - 赤い皿の枚数: 5
   - 金の皿の枚数: 3

3



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


✅ Q19 | 預測:     11 | 正解:     11
     | 題目: 日本の伝統的な行事である「節分」は2月3日です。バレンタインデーは2月14日です。節分の日からバレンタインデーまでは何日
     | raw: 1. 先ず、節分の日の日付を記録しましょう。これは2月3日です。
2. 次に、バレンタインデーの日付を記録しましょう。これは2月14日です。
3. 節分の日からバレンタインデーまでの間隔を計算するために、2月14日 - 2月3日 = 11日

✅ Q20 | 預測:      8 | 正解:      8
     | 題目: 日本の主要4島は本州、北海道、九州、四国です。本州が5つの地方に分けられ、残りの3島がそれぞれ1つの地方であるとした場合
     | raw: 1. 首都圏は本州の一つの地方です。
2. 本州は5つの地方に分けられます。
3. 北海道は1つの地方です。
4. 九州は1つの地方です。
5. 四国は1つの地方です。
6. 主要4島には合計で5 + 1 + 1 + 1 = 8地方がありま

🎯 WizardMath-7B（數學模型） 正確率: 9/20 (45.0%)


In [ ]:
del wizard_model, wizard_tokenizer
gc.collect()
torch.cuda.empty_cache()
print("已釋放 WizardMath-7B 記憶體！")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv

已釋放 WizardMath-7B 記憶體！
memory.used [MiB], memory.free [MiB]
542 MiB, 39900 MiB


### 🔍 Base Model Baseline

記下兩個 base model 的正確率，之後合併模型時可以做比較：
- shisa-gamma-7b：擅長日文，但數學推理能力可能不足
- WizardMath-7B：擅長數學推理，但可能無法理解日文題意

理想的 merge 應該要能結合兩者的優勢！

# **Section 2: Model Merging — 合併模型**

## 2.1 撰寫 Merge 設定檔

以下是一個最簡單的 **50/50 Linear Merge**（線性插值合併）設定。

### ⚠️ 這是你的主要修改區域！

你可以嘗試不同的合併策略，例如：
- **linear**：最直覺的加權平均（調整 `weight` 比例）
- **slerp**：球面線性插值（調整 `t` 參數）
- **ties**：TIES-Merging（需設定 `density` 與 `weight`）
- **dare_ties**：DARE + TIES（需設定 `density`、`weight`）

請參考 [mergekit 文件](https://github.com/arcee-ai/mergekit?tab=readme-ov-file#merge-methods) 了解更多合併方法與參數。

In [24]:
############################################################
# ⚠️ 【可修改區域 — TODO】
#
# 預設：Slerp Merge
# 修改底下 config 參數內容，可以更改之部分與詳情請見：
#
# 可嘗試的方向：
#   1. 調整 weight 比例（例如 0.7:0.3）
#   2. 改用 linear 方法
#   3. 改用 ties 方法
#   4. 改用 dare_ties 方法
#   5. 針對不同 layer 使用不同參數
#
# 提示：每次修改後需重新執行 Section 2 & 3

config = """
models:
  - model: WizardLMTeam/WizardMath-7B-V1.1
    parameters:
      density: 0.7
      weight:
        - filter: self_attn
          value: [0.3, 0.5, 0.7]
        - filter: mlp
          value: [0.3, 0.5, 0.7]
        - value: 0.5
merge_method: dare_ties
base_model: augmxnt/shisa-gamma-7b-v1
parameters:
  rescale: false
dtype: float16
""".strip()


############################################################

with open("merge_config.yml", "w") as f:
    f.write(config)
print("設定檔寫入完成！內容如下：")
print("=" * 40)
print(config)
print("=" * 40)

設定檔寫入完成！內容如下：
models:
  - model: WizardLMTeam/WizardMath-7B-V1.1
    parameters:
      density: 0.7
      weight:
        - filter: self_attn
          value: [0.3, 0.5, 0.7]
        - filter: mlp
          value: [0.3, 0.5, 0.7]
        - value: 0.5
merge_method: dare_ties
base_model: augmxnt/shisa-gamma-7b-v1
parameters:
  rescale: false
dtype: float16


In [25]:
# 備存 merge_config.yml 到 Google Drive
import shutil, os
drive_backup_dir = "/content/drive/MyDrive/ML2026/HW7"
os.makedirs(drive_backup_dir, exist_ok=True)
shutil.copy("merge_config.yml", os.path.join(drive_backup_dir, "merge_config.yml"))
print(f"✅ merge_config.yml 已備存至 {drive_backup_dir}/merge_config.yml")

✅ merge_config.yml 已備存至 /content/drive/MyDrive/ML2026/HW7/merge_config.yml


## 2.2 執行合併

使用 mergekit 進行模型合併（GPU 模式 + lazy unpickle 逐層載入以節省記憶體）。

由於模型大小為 7B 偏大，這個步驟根據不同方法可能需要花上30分鐘至數小時，請耐心等候。

In [26]:
# 若先前有合併過的模型，先清除
!rm -rf ./merged_model

# 執行合併（GPU 模式 + lazy unpickle 逐層載入）
!mergekit-yaml merge_config.yml ./merged_model \
    --lazy-unpickle \
    --allow-crimes \
    --cuda

`torch_dtype` is deprecated! Use `dtype` instead!
Warmup loader cache:   0% 0/2 [00:00<?, ?it/s]

Fetching 12 files: 100% 12/12 [00:00<00:00, 127745.30it/s]

Warmup loader cache:  50% 1/2 [00:00<00:00,  2.06it/s]



Download complete: : 0.00B [00:00, ?B/s]
Fetching 10 files: 100% 10/10 [00:00<00:00, 24513.76it/s]


Warmup loader cache: 100% 2/2 [00:00<00:00,  2.07it/s]
Executing graph: 100% 1457/1457 [01:36<00:00, 15.07it/s]
Fetching 12 files: 100% 12/12 [00:00<00:00, 156796.41it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Download complete: : 0.00B [00:00, ?B/s]
Fetching 12 files: 100% 12/12 [00:00<00:00, 144216.76it/s]

Download complete: : 0.00B [00:00, ?B/s]
Download complete: : 0.00B [01:38, ?B/s]


## 2.3 確認合併輸出

In [27]:
import os

output_dir = "./merged_model"
if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    total_size = sum(os.path.getsize(os.path.join(output_dir, f)) for f in files)
    print(f"\n✅ 合併完成！檔案數: {len(files)}, 總大小: {total_size / 1e9:.2f} GB")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(output_dir, f))
        print(f"  {f}: {size / 1e6:.1f} MB")
else:
    print("❌ 合併失敗，請檢查上方錯誤訊息")


✅ 合併完成！檔案數: 12, 總大小: 14.49 GB
  README.md: 0.0 MB
  added_tokens.json: 0.0 MB
  config.json: 0.0 MB
  mergekit_config.yml: 0.0 MB
  model-00001-of-00003.safetensors: 4886.5 MB
  model-00002-of-00003.safetensors: 4915.9 MB
  model-00003-of-00003.safetensors: 4681.0 MB
  model.safetensors.index.json: 0.0 MB
  special_tokens_map.json: 0.0 MB
  tokenizer.json: 1.8 MB
  tokenizer.model: 0.5 MB
  tokenizer_config.json: 0.0 MB


# **Section 3: Inference — 測試合併後的模型**

## 3.1 載入合併後的模型

In [28]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import re

# Install sentencepiece if not already installed, as it's often needed for tokenizers.
!pip install -U sentencepiece tiktoken
output_dir = "./merged_model"

print("載入合併後的模型中...")
tokenizer = AutoTokenizer.from_pretrained(output_dir)
model = AutoModelForCausalLM.from_pretrained(
    output_dir, torch_dtype=torch.float16, device_map="auto"
)
print("載入完成！")

prompt = "東京の人口は"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50, do_sample=False)
print(f"測試輸出: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

載入合併後的模型中...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


載入完成！
測試輸出: 東京の人口は、2020年には23区の人口が6,972,000人となり、1950年の人口を超えると予測されています


## 3.2 在日文數學題上測試合併模型

使用與 Section 1 相同的題目與評估方式。
Inference 時間可能會需要1-2小時，請耐心等待

In [29]:
import json
import re

# 1. 載入題目
with open("/content/jp_math_question.json", "r", encoding="utf-8") as f:
    questions = json.load(f)

############################################################
# 2. Prompt 模板
USER_PROMPT_TEMPLATE = (
    "次の数学の問題を解いて、最終的な数値の答えを提示してください。\n\n"
    "問題: {question}\n\n"
    "ステップバイステップで考えて、最後に『答え: [数値]』の形式で答えを提示してください。"
)

# ⛔ 嚴禁修改 USER_PROMPT_TEMPLATE！
# 本作業聚焦於 Model Merging 技術本身，
# 而非 Prompt Engineering，因此為確保所有同學的
# 評測條件一致，無論任何情況皆請勿修改以下 prompt。
# 違反此規則將視為違規。
############################################################

# 3. 推論 + 評分
def extract_number(text):
    match = re.search(r'答え\s*[:：]\s*(\d+)', text)
    if match:
        return match.group(1)
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else None

correct = 0
results = []

print(f"{'='*60}")
print("📊 評估合併模型 (Merged Model)")
print(f"{'='*60}\n")

for i, item in enumerate(questions):
    prompt = USER_PROMPT_TEMPLATE.format(question=item["question"])

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            repetition_penalty=1.1
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer_part = full_output[len(prompt):].strip()
    predicted = extract_number(answer_part)
    expected = item["answer_number"]
    is_correct = predicted == expected

    if is_correct:
        correct += 1

    results.append({
        "id": i + 1,
        "correct": is_correct,
        "predicted": predicted,
        "expected": expected,
        "raw": answer_part,
    })

    mark = "✅" if is_correct else "❌"
    print(f"{mark} Q{i+1:02d} | 預測: {str(predicted):>6s} | 正解: {expected:>6s}")
    print(f"     | 題目: {item['question'][:60]}")
    print(f"     | raw: {answer_part[:120]}")
    print()

# 4. 總結
print(f"{'='*50}")
print(f"🎯 Merged Model 正確率: {correct}/{len(questions)} ({correct/len(questions)*100:.1f}%)")
print(f"{'='*50}")

# ==========================================
# 5. 儲存結果為 JSON
# ==========================================
submission_path = "/content/submission.json"

# 若你只想保留 id 和 raw
submission_data = [
    {
        "id": r["id"],
        "raw": r["raw"]
    }
    for r in results
]

with open(submission_path, "w", encoding="utf-8") as f:
    json.dump(submission_data, f, ensure_ascii=False, indent=2)

print(f"\n📁 結果已儲存至 {submission_path}")

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


📊 評估合併模型 (Merged Model)



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


✅ Q01 | 預測:     27 | 正解:     27
     | 題目: 太郎は平成5年に生まれました。次郎は令和2年に生まれました。太郎は次郎より何歳年上ですか？
     | raw: 1. 平成5年と令和2年の差を計算します。
   平成5年 = 1993年
   令和2年 = 2020年
    差 = 2020 - 1993 = 27

2. 太郎が次郎より何歳年上かを求めるには、太郎の生年を令和2年から引きます。




Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

❌ Q17 | 預測:   1892 | 正解:   2036
     | 題目: 2024年は辰年（龍）です。干支は12年で一周します。次に辰年が来るのは西暦何年ですか？
     | raw: 1. 今年は2023年です。
2. 2024年は辰年です。
3. 辰年は12年ごとに来ます。
4. 2024年から12年前は2012年です。
5. 2012年は辰年です。
6. 2012年から12年前は2000年です。
7. 2000年は辰



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


✅ Q18 | 預測:   1400 | 正解:   1400
     | 題目: 回転寿司で、赤い皿は100円、金の皿は300円です。ある客が赤い皿を5枚、金の皿を3枚食べました。消費税を計算する前の合
     | raw: 1. まず、赤い皿の合計金額を計算します。赤い皿は100円で、5枚食べたので、合計金額は100 * 5 = 500円です。
2. 次に、金の皿の合計金額を計算します。金の皿は300円で、3枚食べたので、合計金額は300 * 3 = 900円



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


✅ Q19 | 預測:     11 | 正解:     11
     | 題目: 日本の伝統的な行事である「節分」は2月3日です。バレンタインデーは2月14日です。節分の日からバレンタインデーまでは何日
     | raw: 1. 節分の日は2月3日です。
2. バレンTAINE'S DAYは2月14日です。
3. 節分の日からバレンTAINE'S DAYまでの日数を計算するには、2月14日から2月3日までの日数を引く必要があります。
4. 2月14日から2月3

✅ Q20 | 預測:      8 | 正解:      8
     | 題目: 日本の主要4島は本州、北海道、九州、四国です。本州が5つの地方に分けられ、残りの3島がそれぞれ1つの地方であるとした場合
     | raw: 1. 本州は5つの地方に分けられることがわかっています。
2. 北海道, 九州, 四国はそれぞれ1つの地方であることもわかっています。
3. 合計で4つの島があります (本州, 北海道, 九州, 四国)。
4. 各島には1つの地方があります

🎯 Merged Model 正確率: 13/20 (65.0%)

📁 結果已儲存至 /content/submission.json


In [30]:
# 備存 submission.json 到 Google Drive
import shutil, os
drive_backup_dir = "/content/drive/MyDrive/ML2026/HW7"
os.makedirs(drive_backup_dir, exist_ok=True)
shutil.copy(submission_path, os.path.join(drive_backup_dir, "submission.json"))
print(f"✅ submission.json 已備存至 {drive_backup_dir}/submission.json")

✅ submission.json 已備存至 /content/drive/MyDrive/ML2026/HW7/submission.json
